# The Semi-Grand Canonical Ensemble

In this ensemble we set constant chemical potentials for the species. The number of particles still remains constant.

The first steps are identical as with the Canonical ensemble. We load the model, build the supercell and set up the calculator.

In [ ]:
from icet.core.cluster_expansion import ClusterExpansion
from pathlib import Path
import numpy as np
from ase.build import make_supercell
from mchammer.calculators import ClusterExpansionCalculator

# First, we load the desired model
chemical_symbols = ["Cu", "Ni"]

max_atom_num = 8

base_path = Path.cwd().parents[1]
struct_path = (
    base_path
    / "data"
    / f"CE_dataset_{chemical_symbols[0]}{chemical_symbols[1]}"
)

ce = ClusterExpansion.read(struct_path / f"ce_model_{max_atom_num}.ce")


structure = make_supercell(
    ce.primitive_structure, 3 * np.array([[-1, 1, 1], [1, -1, 1], [1, 1, -1]])
)

temperature = 300
calculator = ClusterExpansionCalculator(structure, ce)


Here, we set up the semi-grand canonical ensemble.

TASK:

- Execute the following for a number of temperatures and chemical potentials. Start at values from -0.5 to 0.5 for the chemical potential.
- Where do you encounter the miscibility gap, does it become smaller at higher temperatures?

In [ ]:
from mchammer.ensembles import SemiGrandCanonicalEnsemble

temperatures = [300]
chemical_potentials = [-0.5]
# Perform the simulations for multiple temperatures and chemical potentials
for temperature in temperatures:
    for dmu in chemical_potentials:

        fname = struct_path / f"sgc-T{temperature}-dmu{dmu:+.3f}.dc"
        # only do this if we did not compute it already
        if not fname.exists():
            mc = SemiGrandCanonicalEnsemble(
                structure=structure,
                calculator=calculator,
                temperature=temperature,
                dc_filename=fname,
                chemical_potentials={
                    chemical_symbols[0]: 0,
                    chemical_symbols[1]: dmu,
                },
            )

            mc.run(number_of_trial_steps=len(structure) * 50)
            structure = mc.structure

In [ ]:
from mchammer import DataContainer
import matplotlib.pyplot as plt
from scipy.integrate import cumulative_trapezoid

n_atoms = len(structure)

for temperature in temperatures:
    energies = []
    concentrations = []
    free_energy_derivatives = []
    for dmu in chemical_potentials:
        equilibration = 10 * n_atoms
        fname = struct_path / f"sgc-T{temperature}-dmu{dmu:+.3f}.dc"
        dc = DataContainer.read(str(fname))
        occupations = dc.get("occupations", start=equilibration)

        energies.append(
            dc.get("potential", start=equilibration) / len(structure)
        )
        concs = []
        for occupation in occupations:
            concs.append(
                np.sum(occupation.symbols == chemical_symbols[1])
                / len(occupation)
            )
        concentrations.append(concs)

        free_energy_derivatives.append(
            dc.ensemble_parameters[f"mu_{chemical_symbols[1]}"]
            - dc.ensemble_parameters[f"mu_{chemical_symbols[0]}"]
        )
    concentrations = np.asarray(concentrations)
    mean_concentrations = concentrations.mean(axis=1)
    free_energy = cumulative_trapezoid(
        free_energy_derivatives,
        mean_concentrations, initial=0
    )
    plt.figure(1)
    plt.scatter(
        concentrations.ravel(),
        np.asarray(energies).ravel(),
        s=2.5,
        label=f"{temperature} K",
    )
    plt.figure(2)
    plt.plot(
        np.asarray(concentrations).mean(axis=1),
        np.asarray(free_energy_derivatives).ravel(),
        "o-",
        label=f"{temperature} K",
    )
    plt.figure(3)
    plt.plot(
        mean_concentrations,
        free_energy,
        "o-",
        label=f"{temperature} K",
    )


plt.figure(1)
plt.ylabel("energy / eV/atom")
plt.xlabel(f"{chemical_symbols[1]} fraction")
plt.legend()
plt.figure(2)
plt.ylabel(f"Free energy derivative / eV/atom")
plt.xlabel(f"{chemical_symbols[1]} fraction")
plt.legend()
plt.figure(3)
plt.xlabel(f"{chemical_symbols[1]} fraction")
plt.ylabel(f"Free energy / eV/atom")
plt.legend()